In [ ]:
# 1. Instalar Ollama
print("1. Descargando e instalando Ollama...")
# Silenciamos la salida detallada de la instalación
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

# 2. Configurar Ollama para que escuche en todas las interfaces
print("\n2. Configurando Ollama para escuchar en el host 0.0.0.0...")
%env OLLAMA_HOST=0.0.0.0

# 3. Lanzar el servidor de Ollama en segundo plano
print("\n3. Lanzando el servidor de Ollama en segundo plano...")
# Este comando ya redirige su salida a ollama.log, así que no necesita cambios.
!nohup ollama serve > ollama.log 2>&1 &

# Dar un momento para que el servidor de Ollama se inicie
print("Esperando 10 segundos para que Ollama se inicie...")
import time
time.sleep(10)

# Verificar si Ollama está funcionando (opcional)
# Si quieres que la verificación sea silenciosa también, descomenta las líneas silenciadas.
print("\nVerificando el estado de Ollama:")
!ollama --version
# !ollama --version > /dev/null 2>&1 # Versión silenciada
# !curl http://localhost:11434/api/tags > /dev/null 2>&1 # Versión silenciada

# 4. Descargar los modelos de embeddings necesarios
print("\n4. Descargando el modelo nomic-embed-text...")
# Silenciamos la barra de progreso de la descarga
!ollama pull nomic-embed-text > /dev/null 2>&1

print("\n5. Descargando el modelo qwen:3-embedding-0.6b...")
# Silenciamos la barra de progreso de la descarga
!ollama pull dengcao/Qwen3-Embedding-8B:Q5_K_M > /dev/null 2>&1

print("\n6. Descargando el modelo Qwen3 8B LLM...")
# En Ollama, el modelo se identifica como `qwen:8b`
!ollama pull qwen3:8b > /dev/null 2>&1


print("\n¡Ollama y los modelos de embeddings están listos!")
print("\nPuedes revisar el log de Ollama con: !cat ollama.log")

In [ ]:
# 3. Lanzar el servidor de Ollama en segundo plano
print("\n3. Lanzando el servidor de Ollama en segundo plano...")
# Este comando ya redirige su salida a ollama.log, así que no necesita cambios.
!nohup ollama serve > ollama.log 2>&1 &

In [ ]:
import importlib
import sys
import subprocess

def verificar_e_instalar(libreria):
  """
  Verifica si una librería está instalada, si no, imprime un mensaje para instalarla.
  """
  try:
    importlib.import_module(libreria)
    print(f"La librería '{libreria}' ya está instalada.")
  except ImportError:
    print(f"Instalando la librería '{libreria}'...")
    # En un entorno real, aquí iría el código para instalar la librería,
    # por ejemplo, usando subprocess para llamar a pip.
    subprocess.check_call([sys.executable, "-m", "pip", "install", libreria])

# Lista de librerías a verificar
librerias = [
    "langchain",
    "langchain_google_genai",
    "langchain_ollama" ,
    "langchain_mongodb",
    "langchain_chroma",
    "langchain_huggingface",
    "sentence_transformers",
    "ragas",
    "matplotlib",
    "seaborn",
    "pymupdf",
    "deepeval",
    "transformers",
    "torch",
    "codecarbon"]

# Verificar cada librería en la lista
for lib in librerias:
  verificar_e_instalar(lib)

Aqui esta el codigo de prueba

In [ ]:
from typing import Any, List, Optional
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import BaseMessage
from langchain_core.outputs import ChatResult
from transformers import AutoTokenizer
from codecarbon import OfflineEmissionsTracker
import time

qwen_tokenizer = None

class MonitoredQwen(BaseChatModel):
    """
    Un contenedor que mide latencia, tokens y energía, y almacena las
    métricas en la variable 'last_call_metrics'.
    """
    llm: BaseChatModel
    last_call_metrics: dict = {}

    def _generate(
        self, messages: List[BaseMessage], stop: Optional[List[str]] = None, **kwargs: Any
    ) -> ChatResult:
        global qwen_tokenizer
        if not qwen_tokenizer:
             qwen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-8B")

        tracker = OfflineEmissionsTracker(country_iso_code="ESP", log_level="error")
        tracker.start()
        start_time = time.time()

        result = self.llm._generate(messages, stop=stop, **kwargs)

        end_time = time.time()
        emissions_data = tracker.stop()

        latency_seconds = end_time - start_time
        energy_consumed_wh = tracker.final_emissions_data.energy_consumed * 1000

        input_text = " ".join([msg.content for msg in messages])
        input_tokens = len(qwen_tokenizer.encode(input_text))
        output_text = result.generations[0].message.content
        output_tokens = len(qwen_tokenizer.encode(output_text))
        total_tokens = input_tokens + output_tokens

        tokens_per_second = total_tokens / latency_seconds if latency_seconds > 0 else 0
        wh_per_token = energy_consumed_wh / total_tokens if total_tokens > 0 else 0
        energy_for_1m_tokens_wh = wh_per_token * 1_000_000 if wh_per_token > 0 else 0

        # ¡CLAVE! Solo almacenamos los datos, ya no los imprimimos aquí.
        self.last_call_metrics = {
            "latency_seconds": latency_seconds,
            "energy_consumed_wh": energy_consumed_wh,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "total_tokens": total_tokens,
            "tokens_per_second": tokens_per_second,
            "wh_per_token": wh_per_token,
            "energy_for_1m_tokens_wh": energy_for_1m_tokens_wh,
            "emissions_kg_co2": emissions_data
        }

        return result

    @property
    def _llm_type(self) -> str:
        return "monitored-qwen"

## Configuración de las Variables de Entorno


In [ ]:
from google.colab import userdata
import os

# Carga tus claves de API desde Colab Secrets
try:
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
    print("GOOGLE_API_KEY loaded successfully from Colab Secrets.")
    # **Verifica que la clave se ha cargado correctamente (solo para depuración)**
    # print(f"First few chars of GOOGLE_API_KEY: {os.environ['GOOGLE_API_KEY'][:5]}...")
except Exception as e:
    print(f"Error loading GOOGLE_API_KEY from Colab Secrets: {e}. Make sure it's set.")


print("Cargando credenciales de MongoDB Atlas desde Colab Secrets...")

try:
    # Cargar el usuario de MongoDB
    os.environ["MONGO_DB_USER"] = userdata.get('MONGO_DB_USER')
    print("✅ MONGO_DB_USER cargado exitosamente.")

    # Cargar la contraseña de MongoDB
    os.environ["MONGO_DB_PASSWORD"] = userdata.get('MONGO_DB_PASSWORD')
    print("✅ MONGO_DB_PASSWORD cargado exitosamente.")

    # Cargar la URI del Cluster de MongoDB
    os.environ["MONGO_DB_CLUSTER_URI"] = userdata.get('MONGO_DB_CLUSTER_URI')
    print("✅ MONGO_DB_CLUSTER_URI cargado exitosamente.")

    print("\n¡Todas las credenciales han sido cargadas en el entorno!")

except Exception as e:
    print(f"\n❌ Error al cargar las credenciales desde Colab Secrets: {e}")
    print("   Asegúrate de haber creado los secretos 'MONGO_DB_USER', 'MONGO_DB_PASSWORD' y 'MONGO_DB_CLUSTER_URI' y haberles dado acceso a este notebook.")


## Interfaz para los Módulos de Vectorización



In [ ]:
from langchain_ollama import OllamaEmbeddings

def myNomicEmbedder():
    embedder = OllamaEmbeddings(model="nomic-embed-text")
    return embedder

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

def myGoogleEmbedder():
    embedder = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
    return embedder

In [ ]:
from langchain_ollama import OllamaEmbeddings

def myQwenEmbedder():
    embedder = OllamaEmbeddings(model="dengcao/Qwen3-Embedding-8B:Q5_K_M")
    return embedder

In [ ]:
class MyEmbedder:
    def __init__(self, model: str):
        self.model = model
        self.dimension = {
            "nomic-embed-text": 768,
            "gemini-embedding-001": 3072,  # Dimensión para Gemini
            "qwen3-embedding": 4096  # Dimensión para Qwen
        }.get(model, None)

    def get_embedder(self):
        if self.model == "nomic-embed-text":
            return myNomicEmbedder()
        elif self.model == "gemini-embedding-001":
            return myGoogleEmbedder()
        elif self.model == "qwen3-embedding":
            return myQwenEmbedder()
        else:
            raise ValueError(f"Model {self.model} not supported")

In [ ]:
#Prueba de los embeddings
embedding_model = 'qwen3-embedding'
embedder = MyEmbedder(model=embedding_model).get_embedder()

# Test the embedder
input_text = 'The meaning of life is 42'
embedding = embedder.embed_query(input_text)
print(f'Lenght of the vector: {len(embedding)}.\nFirst 3 elements: {embedding[:3]} ...')

## Interfaz para los Módulos de Bases de Datos de Vectores

In [ ]:
## ----------------------------------------------------------------------------
## Interfaz para los Módulos de Bases de Datos de Vectores
# Chroma Vector Store - LOCAL
## ----------------------------------------------------------------------------


from langchain_chroma import Chroma

def myChromaDbVectorStore(Embedder: MyEmbedder):
    vector_store = Chroma(
        collection_name=f'collection-{Embedder.model}',
        embedding_function=Embedder.get_embedder(),
        persist_directory="../ChromaDb",
        # Available functions: 'l2', 'cosine', and 'ip'
        collection_metadata={"hnsw:space": "cosine"}
    )
    return vector_store

In [ ]:
## ----------------------------------------------------------------------------
## Interfaz para los Módulos de Bases de Datos de Vectores
# MongoDB Atlas Vector Store - CLOUD
## ----------------------------------------------------------------------------

import os
from pymongo import MongoClient
from langchain_mongodb import MongoDBAtlasVectorSearch

def myMongoDbAtlasVectorStore(Embedder: MyEmbedder, create_index: bool = True):
    client = MongoClient(os.getenv("MONGO_DB_CLUSTER_URI"))

    DB_NAME = "myMongoDbAtlasVectorStore"
    COLLECTION_NAME = f"collection-{Embedder.model}"
    ATLAS_VECTOR_SEARCH_INDEX_NAME = f"index-{Embedder.model}"

    MONGODB_COLLECTION = client[DB_NAME][COLLECTION_NAME]

    vector_store = MongoDBAtlasVectorSearch(
        collection=MONGODB_COLLECTION,
        embedding=Embedder.get_embedder(),
        index_name=ATLAS_VECTOR_SEARCH_INDEX_NAME,
        # Available functions: 'euclidean', 'cosine', and 'dotProduct'
        relevance_score_fn="cosine",
    )

    # Create the vector search index - Needs to be done on the initial setup
    if create_index:
        vector_store.create_vector_search_index(dimensions=Embedder.dimension, filters=["source", "parser"])

    return vector_store

In [ ]:
## ----------------------------------------------------------------------------
## INTERFAZ PARA LOS MÓDULOS DE BASES DE DATOS DE VECTORES:
# Interfaz personalizada para definir de manera dinámica el modulo de base
# de datos de vectores.

# Proveedores Disponibles:
# - chromadb
# - mongodb
## ----------------------------------------------------------------------------

from uuid import uuid4

class MyVectorStore:

    def __init__(self, provider: str, Embedder: MyEmbedder):
        self.provider = provider
        self.Embedder = Embedder

    def get_vector_store(self):
        if self.provider == "chromadb":
            return myChromaDbVectorStore(Embedder=self.Embedder)

        elif self.provider == "mongodb":
            return myMongoDbAtlasVectorStore(Embedder=self.Embedder)
        else:
            raise ValueError(f"Provider {self.provider} not supported")

    def get_vector_store_documents(self, source: str, parser: str):
        vector_store = self.get_vector_store()

        if self.provider == "chromadb":
            old_docs = vector_store.get(where={"$and": [{"source": source}, {"parser": parser}]})
            return old_docs["ids"]

        elif self.provider == "mongodb":
            old_docs = list(vector_store._collection.find({"source": source, "parser": parser}, {"_id": 1}))
            return [doc["_id"] for doc in old_docs]

    def purge_vector_store_documents(self, source: str, parser: str):
        vector_store = self.get_vector_store()
        document_ids = self.get_vector_store_documents(source=source, parser=parser)
        if document_ids:
            vector_store.delete(ids=document_ids)

    def push_vector_store_documents(self, documents: list, embeddings: list = None):
        vector_store = self.get_vector_store()
        if not embeddings:
            vector_store.add_documents(documents=documents)
            return

        if self.provider == "chromadb":
            vector_store._collection.add(
                ids=[str(uuid4()) for _ in range(len(documents))],
                documents=[doc.page_content for doc in documents],
                embeddings=embeddings,
                metadatas=[doc.metadata for doc in documents]
            )

        elif self.provider == "mongodb":
            vector_store._collection.insert_many([
                {
                    "text" : text,
                    "embedding" : embedding,
                    **metadata
                } for text, embedding, metadata in zip(
                    [doc.page_content for doc in documents],
                    embeddings,
                    [doc.metadata for doc in documents]
                )
            ])

In [ ]:
## ----------------------------------------------------------------------------
## INTERFAZ PARA LOS MÓDulos DE RETRIEVAL:
# Interfaz personalizada para definir de manera dinámica el modulo de
# retrieval.
## ----------------------------------------------------------------------------


from langchain.retrievers.multi_query import MultiQueryRetriever

class MyRetriever:

    # <--- CAMBIO: Añadimos retrieval_strategy y parámetros para el LLM del retriever
    def __init__(self, k: int, source: str, parser: str, VectorStore: MyVectorStore,
                 retrieval_strategy: str = "similarity",
                 llm_model: str = "gemini-2.5-flash",
                 temperature: float = 0.0):
        self.k = k
        self.source = source
        self.parser = parser
        self.VectorStore = VectorStore
        self.retrieval_strategy = retrieval_strategy # <--- CAMBIO
        self.llm_model = llm_model                   # <--- CAMBIO
        self.temperature = temperature               # <--- CAMBIO

    def get_retriever(self):
        vector_store = self.VectorStore.get_vector_store()

        # --- Estrategia 1: Búsqueda por similitud (Comportamiento Original) ---
        if self.retrieval_strategy == "similarity":
            print("Estrategia de Retrieval: Similitud Estándar")
            if self.VectorStore.provider == "chromadb":
                return vector_store.as_retriever(search_type='similarity', search_kwargs={'k': self.k, 'filter': {"$and": [{"source": self.source}, {"parser": self.parser}]}})
            elif self.VectorStore.provider == "mongodb":
                return vector_store.as_retriever(search_type='similarity', search_kwargs={'k': self.k, 'pre_filter': {"source": {"$eq": self.source}, "parser": {"$eq": self.parser}}})


        elif self.retrieval_strategy == "multiquery":
            print(f"Estrategia de Retrieval: Multi-Query con LLM '{self.llm_model}'")
            # Creamos el retriever base sobre el que actuará MultiQuery
            if self.VectorStore.provider == "chromadb":
                base_retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k': self.k, 'filter': {"$and": [{"source": self.source}, {"parser": self.parser}]}})
            elif self.VectorStore.provider == "mongodb":
                base_retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k': self.k, 'pre_filter': {"source": {"$eq": self.source}, "parser": {"$eq": self.parser}}})
            else:
                 raise ValueError(f"Provider {self.provider} not supported for MultiQuery base retriever")

            llm_for_retriever = MyLLM(model=self.llm_model, temperature=self.temperature).get_llm()

            multi_query_retriever = MultiQueryRetriever.from_llm(
                retriever=base_retriever, llm=llm_for_retriever
            )
            return multi_query_retriever

        else:
            raise ValueError(f"Retrieval strategy '{self.retrieval_strategy}' not supported. Opciones válidas: 'similarity', 'multiquery'.")

In [ ]:
## ----------------------------------------------------------------------------
## INTERFAZ PARA LOS MÓDULOS DE BASES DE DATOS DE VECTORES:
# Prueba de la interfaz personalizada para los módulos de base de
# datos de vectores - Carga de documentos.
## ----------------------------------------------------------------------------

from uuid import uuid4
from langchain_core.documents import Document

# Define the documents to add to the vector store
document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "test", "parser": "test"}
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "test", "parser": "test"}
)

document_3 = Document(
    page_content="There has been a significant increase in the number of COVID-19 cases in the past week.",
    metadata={"source": "test", "parser": "test"}
)

documents = [document_1, document_2, document_3]
uuids = [str(uuid4()) for _ in range(len(documents))]

# Load the documents into the vector stores
# Available providers: 'chromadb', 'mongodb'
for db_provider in ["chromadb", "mongodb"]:
    # Using all the possible embedding models
    # Available models: 'nomic-embed-text', 'snowflake-arctic-embed-335m', 'text-embedding-3-large', 'voyage-3', 'mistral-embed'
    for embedding_model in ["nomic-embed-text","gemini-embedding-001", "qwen3-embedding"]:

        # Define the Embedder and Vector Store
        Embedder = MyEmbedder(model=embedding_model)
        VectorStore = MyVectorStore(provider=db_provider, Embedder=Embedder)

        # Purge Existing Documents
        VectorStore.purge_vector_store_documents(source="test", parser="test")

        # Embed the Documents
        embedder = Embedder.get_embedder()
        embeddings = [embedder.embed_query(doc.page_content) for doc in documents]

        # Push the Documents
        VectorStore.push_vector_store_documents(documents=documents, embeddings=embeddings)

In [ ]:
## ----------------------------------------------------------------------------
## INTERFAZ PARA LOS MÓDULOS DE RETRIEVAL:
# Prueba de la interfaz personalizada para los módulos de retrieval - Busqueda
# de documentos.
## ----------------------------------------------------------------------------

# Define the Embedder
# Available models: 'nomic-embed-text', 'snowflake-arctic-embed-335m', 'text-embedding-3-large', 'voyage-3', 'mistral-embed'
embedding_model = "qwen3-embedding"
Embedder = MyEmbedder(model=embedding_model)

# Define the Vector Store
# Available providers: 'chromadb', 'mongodb'
db_provider = "mongodb"
VectorStore = MyVectorStore(provider=db_provider, Embedder=Embedder)

# Define the Retriever and its filters
k, source, parser = 2, "test", "test"
retriever = MyRetriever(k=k, source=source, parser=parser, VectorStore=VectorStore).get_retriever()

# Test the retriever
query = "Will it be hot tomorrow?"
docs = retriever.invoke(query)
print(docs[0].page_content)

## Interfaz para los Módulos de LLMs

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

def gemini_2_5_flash(temperature: float):
    return ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=temperature)


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

def gemini_2_5_pro(temperature: float):
    return ChatGoogleGenerativeAI(model="gemini-2.5-pro", temperature=temperature)


In [ ]:
from langchain_ollama import ChatOllama

# La función ahora devuelve el LLM dentro de nuestro wrapper
def qwen3(temperature: float):
    base_llm = ChatOllama(model="qwen3:8b", temperature=temperature)
    return MonitoredQwen(llm=base_llm)

In [ ]:
class MyLLM:

    def __init__(self, model: str, temperature: float):
        self.model = model
        self.temperature = temperature

    def get_llm(self):
        if self.model == "gemini-2.5-flash":
            return gemini_2_5_flash(temperature=self.temperature)
        elif self.model == "gemini-2.5-pro":
            return gemini_2_5_pro(temperature=self.temperature)
        elif self.model == "qwen3-8b":
            return qwen3(temperature=self.temperature)
        else:
            raise ValueError(f"Model {self.model} not supported")

In [ ]:
llm_model, temperature = "qwen3-8b", 1
llm = MyLLM(model=llm_model, temperature=temperature).get_llm()

# Test the LLM
messages = [("human", "What is the meaning of life? en es")]
response = llm.invoke(messages)
print(response.content)

## Interfaz para el Procesamiento de PDFs

En esta sección, construiremos nuestra interfaz personalizada para el procesamiento de PDFs. Su función principal será extraer el contenido de un documento PDF y fragmentarlo en unidades de información, que luego serán cargadas en las bases de datos de vectores. Aunque LangChain proporciona algunos conectores para el procesamiento de PDFs, no todos los conectores están disponibles ni existe una interfaz estándar. Por lo tanto, seremos responsables de crear esta interfaz, homogenizando tanto la entrada como la salida, para asegurar que todos los módulos de procesamiento de PDFs funcionen de manera consistente en nuestros experimentos.

In [ ]:
## ----------------------------------------------------------------------------
## Interfaz para los Módulos de Procesamiento de PDFs
# PyMuPDF Loader - LOCAL
## ----------------------------------------------------------------------------

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

def myPyMuPDFLoader(path, chunk_size=500, chunk_overlap=50):

    loader = PyMuPDFLoader(file_path=path)
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    return text_splitter.split_documents(loader.load())


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter
from langchain_core.documents import Document
import os

def markerprocessor(path, chunk_size=2000, chunk_overlap=200): # Aumentamos un poco el chunk_size por defecto
    nombre_archivo_pdf = os.path.basename(path)
    nombre_base = os.path.splitext(nombre_archivo_pdf)[0]
    ruta_md = os.path.join("/content/drive/MyDrive/Tesis/documents/markerprocessor", f"{nombre_base}.md")

    if not os.path.exists(ruta_md):
        raise FileNotFoundError(f"El archivo Markdown no se encontró en la ruta esperada: {ruta_md}. "
                              "Asegúrate de haber ejecutado primero el script de conversión de Marker.")

    with open(ruta_md, "r", encoding="utf-8") as f:
        contenido_md = f.read()

    # --- PASO 1: División semántica por encabezados (como ya lo tenías) ---
    headers_to_split_on = [
        ("#", "Header 1"),
        ("##", "Header 2"),
        ("###", "Header 3"),
    ]

    markdown_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=headers_to_split_on, strip_headers=False
    )
    semantic_chunks = markdown_splitter.split_text(contenido_md)
    print(f"Cargado '{ruta_md}' en {len(semantic_chunks)} chunks semánticos iniciales.")

    # --- PASO 2: (NUEVO) División por tamaño para asegurar que ningún chunk exceda el límite ---
    # Usamos un text_splitter para dividir cualquier chunk que sea demasiado grande.
    # El límite de Gemini es de aprox 32k caracteres, así que 2000-4000 es un tamaño seguro.
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    # El método split_documents toma una lista de Documentos y la procesa.
    final_chunks = text_splitter.split_documents(semantic_chunks)

    print(f"Dividido en un total de {len(final_chunks)} chunks finales para cumplir con los límites de tamaño.")
    return final_chunks

In [ ]:
# Célula de código con ID: wfSPmhrYTZTA
# Reemplaza el contenido de esta célula con lo siguiente:

from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter
from langchain_core.documents import Document
import os

def geminiprocessor(path, chunk_size=4000, chunk_overlap=200): # Aumentamos el chunk_size por defecto
    """
    Carga un archivo Markdown pre-procesado por 'Gemini' y lo divide en chunks
    combinando la estructura de encabezados con un límite de tamaño.

    Args:
        path (str): La ruta al archivo PDF ORIGINAL. La función inferirá la ruta del .md.
    """
    # La lógica de derivación es idéntica, solo cambia el nombre de la carpeta
    nombre_base = os.path.splitext(os.path.basename(path))[0]
    ruta_md = os.path.join("/content/drive/MyDrive/Tesis/documents/geminiprocessor", f"{nombre_base}.md")

    if not os.path.exists(ruta_md):
        raise FileNotFoundError(f"El archivo Markdown no se encontró en la ruta esperada: {ruta_md}. "
                              "Asegúrate de haber ejecutado primero el script de conversión de Gemini.")

    with open(ruta_md, "r", encoding="utf-8") as f:
        contenido_md = f.read()

    # --- PASO 1: División semántica por encabezados ---
    headers_to_split_on = [
        ("#", "Header 1"),
        ("##", "Header 2"),
        ("###", "Header 3"),
    ]

    markdown_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=headers_to_split_on, strip_headers=False
    )
    semantic_chunks = markdown_splitter.split_text(contenido_md)
    print(f"Cargado '{ruta_md}' en {len(semantic_chunks)} chunks semánticos iniciales.")

    # --- PASO 2: (NUEVO Y CRUCIAL) División por tamaño para asegurar que ningún chunk exceda el límite ---
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, # Un tamaño seguro, muy por debajo de 36k
        chunk_overlap=chunk_overlap
    )

    # El método split_documents toma una lista de Documentos y la procesa.
    final_chunks = text_splitter.split_documents(semantic_chunks)

    print(f"Dividido en un total de {len(final_chunks)} chunks finales para cumplir con los límites de tamaño.")
    return final_chunks

In [ ]:

import json
from langchain_core.output_parsers import JsonOutputParser # Cambiamos el parser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document

## ----------------------------------------------------------------------------
## Interfaz para los Módulos de Procesamiento de PDFs
# agenticgeminiprocessor - CLOUD (Chunking semántico con Gemini)
## ----------------------------------------------------------------------------

def agenticgeminiprocessor(path, chunk_size=500, chunk_overlap=50):
    """
    Carga un archivo Markdown pre-procesado y utiliza el LLM Gemini para dividirlo
    en chunks semánticos y lógicos.
    """
    nombre_base = os.path.splitext(os.path.basename(path))[0]
    ruta_md = os.path.join("/content/drive/MyDrive/Tesis/documents/geminiprocessor", f"{nombre_base}.md") # Asume que usamos los MD de Gemini
    print(ruta_md)
    if not os.path.exists(ruta_md):
        raise FileNotFoundError(f"El archivo Markdown no se encontró en: {ruta_md}. Ejecuta primero el script de conversión.")

    with open(ruta_md, "r", encoding="utf-8") as f:
        contenido_md = f.read()

    llm = MyLLM(model="gemini-2.5-flash", temperature=0).get_llm()

    # --- PROMPT CORREGIDO Y MEJORADO ---
    prompt_template = """
    Actúa como un experto en procesamiento de datos para sistemas RAG. Tu tarea es segmentar el siguiente documento Markdown en chunks lógicos y autocontenidos.
    Cada chunk debe representar una idea, tema o sección completa. No cortes frases ni conceptos a la mitad.

    Devuelve el resultado como un objeto JSON válido con una única clave "chunks", que contiene una lista de strings. Cada string de la lista será un chunk.

    IMPORTANTE: Si el texto de un chunk contiene comillas dobles ("), debes escaparlas con una barra invertida (\\") para que el JSON sea válido.

    Ejemplo de salida:
    {{
      "chunks": [
        "Este es el primer chunk que habla sobre la introducción del tema.",
        "Este es un segundo chunk que detalla la metodología y menciona una \"cita importante\".",
        "Este es el tercer chunk que presenta los resultados principales."
      ]
    }}

    Aquí está el documento que debes procesar:
    ---
    {document_content}
    """

    rag_prompt = ChatPromptTemplate.from_template(prompt_template)

    # --- CADENA MEJORADA CON JSON PARSER ---
    parser = JsonOutputParser()
    chain = rag_prompt | llm | parser

    print(f"Enviando documento a Gemini para chunking semántico...")

    try:
        # El parser ya devuelve un diccionario de Python, no un string
        response_dict = chain.invoke({"document_content": contenido_md})

        text_chunks = response_dict.get("chunks", [])

        if not text_chunks:
            print("Advertencia: El LLM no devolvió chunks. Devolviendo el documento completo.")
            return [Document(page_content=contenido_md)]

        documents = [Document(page_content=chunk) for chunk in text_chunks]
        print(f"Gemini ha generado {len(documents)} chunks semánticos.")

    except Exception as e:
        print(f"Error al invocar la cadena o parsear la respuesta del LLM: {e}")
        print("Devolviendo el documento completo como un solo chunk de fallback.")
        return [Document(page_content=contenido_md)]

    return documents

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:

import re
import json
from langchain_core.output_parsers import StrOutputParser # Cambiamos el parser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document


def agenticqwenprocessor(path, chunk_size=500, chunk_overlap=50):
    """
    Carga un archivo Markdown y utiliza el LLM Qwen para dividirlo en chunks semánticos.
    Esta versión es más robusta para manejar salidas no-JSON de modelos locales.
    """
    nombre_base = os.path.splitext(os.path.basename(path))[0]
    ruta_md = os.path.join("/content/drive/MyDrive/Tesis/documents/markerprocessor", f"{nombre_base}.md")

    if not os.path.exists(ruta_md):
        raise FileNotFoundError(f"El archivo Markdown no se encontró en: {ruta_md}. Ejecuta el script de conversión.")

    with open(ruta_md, "r", encoding="utf-8") as f:
        contenido_md = f.read()

    llm = MyLLM(model="qwen3-8b", temperature=0).get_llm()

    # --- PROMPT DEFINITIVAMENTE CORREGIDO (CON LLAVES ESCAPADAS) ---
    prompt_template = """
    Tu única tarea es segmentar el siguiente documento Markdown en una lista de chunks.
    Devuelve SOLAMENTE un objeto JSON válido, clave "chunks" que contenga una lista de strings.


    DOCUMENTO A PROCESAR:
    ---
    {document_content}
    """

    rag_prompt = ChatPromptTemplate.from_template(prompt_template)

    chain = rag_prompt | llm | StrOutputParser()

    print(f"Enviando documento a Qwen para chunking semántico...")

    try:
        response_str = chain.invoke({"document_content": contenido_md})

        # Lógica de extracción de JSON robusta
        json_match = re.search(r'\{.*\}', response_str, re.DOTALL)

        if not json_match:
            print("Error: No se encontró un objeto JSON en la respuesta del LLM.")
            print("Respuesta recibida:", response_str)
            raise ValueError("No JSON object in LLM response")

        json_str = json_match.group(0)

        # A veces, los LLMs escapan comillas innecesariamente. Un simple replace puede ayudar.
        # Esto es un pequeño truco para limpiar antes de parsear.
        cleaned_json_str = json_str.replace('\\"', '"')

        # Otro problema común es que el LLM añada una coma al final de la lista.
        # Intentamos eliminarla si existe.
        cleaned_json_str = re.sub(r',\s*\]', ']', cleaned_json_str)
        cleaned_json_str = re.sub(r',\s*\}', '}', cleaned_json_str)

        data = json.loads(cleaned_json_str)
        text_chunks = data.get("chunks", [])

        if not text_chunks:
            print("Advertencia: El LLM no devolvió chunks en el JSON. Devolviendo el documento completo.")
            return [Document(page_content=contenido_md)]

        documents = [Document(page_content=chunk) for chunk in text_chunks]
        print(f"Qwen ha generado {len(documents)} chunks semánticos.")

    except Exception as e:
        print(f"Error al invocar la cadena o parsear la respuesta del LLM: {e}")
        # Imprimimos la respuesta completa para facilitar la depuración
        print("--- INICIO RESPUESTA COMPLETA DEL LLM ---")
        print(response_str)
        print("--- FIN RESPUESTA COMPLETA DEL LLM ---")
        print("Devolviendo el documento completo como un solo chunk de fallback.")
        return [Document(page_content=contenido_md)]

    return documents

In [ ]:
## # Test Loader
path = r"/content/drive/MyDrive/documents/Creditos Inteligentes.pdf"
documents = agenticgeminiprocessor(path)
print(f"Loaded {len(documents)} documents from {path}")

In [ ]:
import os

class MyPdfLoader:


    def __init__(self, path: str, loader: str, chunk_size=500, chunk_overlap=50):

        self.path = path
        self.loader = loader
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.source = os.path.basename(self.path)

    def extract(self) -> list:

        # La estructura if/elif/else asegura que solo se ejecute un loader.
        if self.loader == "PyMuPDF":
            docs = myPyMuPDFLoader(self.path, chunk_size=self.chunk_size, chunk_overlap=self.chunk_overlap)

        elif self.loader == "markerprocessor":
            docs = markerprocessor(self.path, chunk_size=self.chunk_size, chunk_overlap=self.chunk_overlap)

        elif self.loader == "geminiprocessor":
            docs = geminiprocessor(self.path, chunk_size=self.chunk_size, chunk_overlap=self.chunk_overlap)

        elif self.loader == "agenticqwenprocessor":
            docs = agenticqwenprocessor(self.path, chunk_size=self.chunk_size, chunk_overlap=self.chunk_overlap)

        elif self.loader == "agenticgeminiprocessor":
            docs = agenticgeminiprocessor(self.path, chunk_size=self.chunk_size, chunk_overlap=self.chunk_overlap)

        else:
           raise ValueError(f"Loader desconocido: '{self.loader}'. Los loaders válidos son: "
    "'PyMuPDF', 'markerprocessor', 'geminiprocessor', 'agenticqwenprocessor', 'agenticgeminiprocessor'"
)
        return docs

    def load(self, VectorStore: 'MyVectorStore', documents: list = None, embeddings: list = None, replace: bool = False):

        document_ids = VectorStore.get_vector_store_documents(source=self.source, parser=self.loader)
        if document_ids:
            if replace:
                print(f"Borrando {len(document_ids)} documentos existentes para source='{self.source}' y parser='{self.loader}'...")
                VectorStore.purge_vector_store_documents(source=self.source, parser=self.loader)
            else:
                print(f"Los documentos para source='{self.source}' y parser='{self.loader}' ya existen. "
                      "Usa replace=True para sobrescribirlos.")
                return

        # Extrae los documentos si no se proporcionan
        if not documents:
            documents = self.extract()

        for document in documents:
            document.metadata.update({"source": self.source, "parser": self.loader})
        print(f"Cargando {len(documents)} nuevos documentos en el VectorStore...")

        VectorStore.push_vector_store_documents(documents=documents, embeddings=embeddings)
        print("Carga completada.")

In [ ]:
# Prueba de la interfaz personalizada para los módulos de procesamiento de PDFs.

path = "/content/drive/MyDrive/Tesis/documents/Creditos Inteligentes.pdf"


embedding_model = "gemini-embedding-001"
Embedder = MyEmbedder(embedding_model)

db_provider = "mongodb"
VectorStore = MyVectorStore(provider=db_provider, Embedder=Embedder)


pdf_loader = "PyMuPDF"
PdfLoader = MyPdfLoader(path=path, loader=pdf_loader)


PdfLoader.load(VectorStore=VectorStore, replace=True)

## Gestión de Carga de Documentos PDF



In [ ]:
import logging
from pathlib import Path
import os

logging.getLogger('requests').setLevel(logging.CRITICAL)
logging.getLogger('urllib3').setLevel(logging.CRITICAL)

# --- RUTAS DE LOS PDFs ---
pdf_files = [
    #"/content/drive/MyDrive/Tesis/documents/Creditos Inteligentes.pdf",
    #"/content/drive/MyDrive/Tesis/documents/Crédito Tradicional.pdf",
    #"/content/drive/MyDrive/Tesis/documents/Manual de Politicas de Crédito.pdf",
    "/content/drive/MyDrive/Tesis/documents/SOP INSTRUCTIVO PARA LA VERIFICACION DE CRÉDITOS.pdf"
]

# Verificar qué PDFs existen
final_pdfs_list = []
print("Verificando PDFs...")
for pdf_path in pdf_files:
    path = Path(pdf_path)
    if path.exists():
        final_pdfs_list.append(path)
        print(f"✓ {path.name}")
    else:
        print(f"✗ No encontrado: {path.name}")

print(f"\nPDFs a procesar: {len(final_pdfs_list)}")

# --- PROCESAMIENTO ---
if final_pdfs_list:
    pdf_loader_list = ["PyMuPDF", "geminiprocessor","markerprocessor", "agenticgeminiprocessor"]
    embedding_models_list = ["nomic-embed-text", "gemini-embedding-001", "qwen3-embedding"]
    db_providers_list = ["chromadb", "mongodb"]

    total = len(final_pdfs_list) * len(pdf_loader_list) * len(embedding_models_list) * len(db_providers_list)
    count = 0

    for path in final_pdfs_list:
        for pdf_loader in pdf_loader_list:
            print(f"Extrayendo {path.name} con {pdf_loader}...")
            PdfLoader = MyPdfLoader(path=path, loader=pdf_loader)
            documents = PdfLoader.extract()

            for embedding_model in embedding_models_list:
                print(f"Embedding {path.name} con {embedding_model}...")
                Embedder = MyEmbedder(embedding_model)
                embeddings = [Embedder.get_embedder().embed_query(doc.page_content) for doc in documents]

                for db_provider in db_providers_list:
                    count += 1
                    print(f"Cargando en {db_provider}... ({count}/{total})")
                    VectorStore = MyVectorStore(provider=db_provider, Embedder=Embedder)
                    PdfLoader.load(VectorStore=VectorStore, documents=documents, embeddings=embeddings, replace=False)

else:
    print("No hay PDFs para procesar")

## Interfaz para el Flujo RAG

En esta sección, definimos la interfaz del flujo RAG, que permite construir y ejecutar fácilmente un conjunto de RAGs. Este flujo toma como entrada los módulos configurados previamente, como el procesador de PDFs, el modelo de embeddings, la base de datos de vectores y el modelo de lenguaje. Además, permite definir parámetros como el número de resultados a recuperar (top k) y la temperatura del modelo de lenguaje para personalizar las respuestas generadas.

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables import RunnableMap


class MyRAGChain:

    def __init__(self, source: str, pdf_loader: str, embedding_model: str, db_provider: str, llm_model: str, k: int = 5, temperature: float = 0.5,
                 retrieval_strategy: str = "similarity",
                 retriever_llm_model: str = "gemini-2.5-flash",
                 retriever_llm_temperature: float = 0.0,
                 llm_instance = None): # <-- El nuevo parámetro está aquí
        self.source = source
        self.pdf_loader = pdf_loader
        self.embedding_model = embedding_model
        self.db_provider = db_provider
        self.llm_model = llm_model
        self.k = k
        self.temperature = temperature
        self.retrieval_strategy = retrieval_strategy
        self.retriever_llm_model = retriever_llm_model
        self.retriever_llm_temperature = retriever_llm_temperature
        self.llm_instance = llm_instance

    def get_rag_chain(self):


        def format_docs(docs):
            return "\n\n".join(doc.page_content for doc in docs)

        RAG_TEMPLATE = """
        You are an assistant for question-answering tasks.
        Use only the following pieces of retrieved context to answer the question.
        If you don't know the answer, just say that you don't know.

        <context>
        {context}
        </context>

        Answer the following question:

        {question}"""
        rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)


        Embedder = MyEmbedder(model=self.embedding_model)
        VectorStore = MyVectorStore(provider=self.db_provider, Embedder=Embedder)

        base_retrieval_strategy_for_hyde = "similarity" if self.retrieval_strategy == "hyde" else self.retrieval_strategy
        retriever = MyRetriever(
            k=self.k,
            source=self.source,
            parser=self.pdf_loader,
            VectorStore=VectorStore,
            retrieval_strategy=base_retrieval_strategy_for_hyde,
            llm_model=self.retriever_llm_model,
            temperature=self.retriever_llm_temperature
        ).get_retriever()


        if self.llm_instance:
            llm = self.llm_instance
        else:
            llm = MyLLM(model=self.llm_model, temperature=self.temperature).get_llm()


        if self.retrieval_strategy == "hyde":
            pass
        else:
            full_chain = (
                {
                    "retrievals": retriever,
                    "question": RunnablePassthrough(),
                }
                | RunnableMap(
                    {
                        "retrievals": lambda inputs: inputs["retrievals"],
                        "answer": (
                            {
                                "context": lambda inputs: format_docs(inputs["retrievals"]),
                                "question": lambda inputs: inputs["question"],
                            }
                            | rag_prompt
                            | llm
                            | StrOutputParser()
                        ),
                    }
                )
            )

        return full_chain | (lambda outputs: {"retrievals": outputs["retrievals"], "answer": outputs["answer"]})

In [ ]:
## ----------------------------------------------------------------------------
## INTERFAZ PARA EL FLUJO RAG:
# Script de prueba robusto que maneja tanto LLMs locales (con métricas)
# como LLMs en la nube (sin métricas).
## ----------------------------------------------------------------------------
import time


# --- 1. Parámetros de Configuración ---
# PUEDES CAMBIAR ESTA LÍNEA ENTRE "qwen3-8b" Y "gemini-2.5-flash" SIN QUE SE ROMPA
llm_model, temperature = "qwen3-8b", 0.5
#llm_model, temperature = "qwen3-8b", 0.5 # <- Descomenta esta para probar el local

embedding_model = "qwen3-embedding"
db_provider = "mongodb"
pdf_loader = "PyMuPDF"
k, source = 5, "Creditos Inteligentes.pdf"
question = "¿Cuáles son los beneficios de utilizar un sistema de créditos inteligentes?"

# --- 2. Creación e Inyección del LLM ---
print(f"Creando instancia del LLM: {llm_model}...")
# Esta variable 'llm_instance' contendrá el objeto del modelo, sea local o de la nube.
llm_instance = MyLLM(model=llm_model, temperature=temperature).get_llm()

print("Construyendo el pipeline RAG...")
rag_chain = MyRAGChain(
    source=source,
    pdf_loader=pdf_loader,
    embedding_model=embedding_model,
    db_provider=db_provider,
    llm_model=llm_model,
    k=k,
    temperature=temperature,
    llm_instance=llm_instance
).get_rag_chain()
print("Pipeline listo.")

# --- 3. Ejecución y Recolección de Datos ---
print(f"\nEjecutando la consulta...")
total_start_time = time.time()
output = rag_chain.invoke(question)
total_end_time = time.time()

# --- ¡LÓGICA CORREGIDA Y ROBUSTA! ---
# Comprobamos si el objeto LLM tiene nuestro atributo de métricas.
# hasattr() es la forma segura de verificar esto en Python.
llm_metrics = {} # Por defecto, un diccionario vacío.
if hasattr(llm_instance, 'last_call_metrics'):
    # Si el atributo existe, significa que usamos 'MonitoredQwen' y podemos obtener los datos.
    llm_metrics = llm_instance.last_call_metrics

# Consolidamos todo en un único objeto de reporte
total_latency = total_end_time - total_start_time
report = {
    "rag_output": output,
    "total_pipeline_latency_seconds": total_latency,
    "llm_metrics": llm_metrics
}

# --- 4. Presentación del Reporte (AHORA ES CONDICIONAL) ---
print("\n========================= REPORTE DE EJECUCIÓN =========================")
print(f"⏱️ Latencia Total del Pipeline: {report['total_pipeline_latency_seconds']:.2f} segundos.")
print("----------------------------------------------------------------------")
print("✅ Respuesta Generada:")
print(report['rag_output']['answer'])
print("----------------------------------------------------------------------")

# ¡CAMBIO CLAVE! Solo mostramos las métricas detalladas si el diccionario no está vacío.
if report['llm_metrics']:
    print("📊 Métricas del LLM Local:")
    # Usamos .get() para mayor seguridad, aunque no sería estrictamente necesario aquí.
    print(f"   - Latencia del LLM: {report['llm_metrics'].get('latency_seconds', 0):.2f} s")
    print(f"   - Tokens de Entrada: {report['llm_metrics'].get('input_tokens', 'N/A')}")
    print(f"   - Tokens de Salida: {report['llm_metrics'].get('output_tokens', 'N/A')}")
    print(f"   - Tokens Totales: {report['llm_metrics'].get('total_tokens', 'N/A')}")
    print(f"   - Velocidad: {report['llm_metrics'].get('tokens_per_second', 0):.2f} Tokens/s")
    print(f"   - Consumo: {report['llm_metrics'].get('energy_consumed_wh', 0):.4f} Wh")
    print(f"   - Eficiencia: {report['llm_metrics'].get('wh_per_token', 0):.6f} Wh/token")
else:
    print("📊 Métricas del LLM Local: No disponibles (se utilizó un modelo no monitoreado).")

print("----------------------------------------------------------------------")
print("📚 Documentos Recuperados:")
for i, doc in enumerate(report['rag_output']['retrievals']):
    print(f"  [{i+1}] {doc.page_content[:150]}...")
print("======================================================================")

## Flujo de Evaluación con DeepEval y RAGAs

En esta sección, definimos el flujo de evaluación, que itera sobre todos nuestros pipelines RAG aplicándolos a las consultas del dataset de evaluación. El objetivo es calcular las métricas RAGAs, que permiten medir el rendimiento de los distintos flujos de RAG en función de las respuestas generadas.

Correcto


In [ ]:
## ----------------------------------------------------------------------------
## FLUJO DE EVALUACIÓN CON DEEPEVAL Y RAGAS:
# Unidad de evaluación (VERSIÓN CORREGIDA Y FINAL)
## ----------------------------------------------------------------------------

import os
import json
import time
from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.models import DeepEvalBaseLLM
from deepeval.metrics import AnswerRelevancyMetric, FaithfulnessMetric, ContextualRecallMetric, ContextualPrecisionMetric

def gemini_evaluate_test_case(RAGChain: MyRAGChain, test_case: dict, evaluation_model: DeepEvalBaseLLM, evaluation_json: str, replace: bool = False):

    try:
        with open(evaluation_json, 'r', encoding='utf-8') as f:
            existing_evaluation_list = json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        existing_evaluation_list = []

    # --- INICIO DE LA MODIFICACIÓN CLAVE ---
    # Hacemos la comprobación de unicidad MÁS específica.
    evaluation_item = [item for item in existing_evaluation_list
                       if item.get('input') == test_case['input']
                       and item.get('llm_model') == RAGChain.llm_model
                       and item.get('embedding_model') == RAGChain.embedding_model
                       and item.get('pdf_loader') == RAGChain.pdf_loader
                       and item.get('db_provider') == RAGChain.db_provider
                       and item.get('retrieval_strategy') == RAGChain.retrieval_strategy
                       # ## <-- ¡ESTA ES LA LÍNEA CRÍTICA QUE FALTABA! ##
                       and item.get('retriever_llm_model') == RAGChain.retriever_llm_model]
    # --- FIN DE LA MODIFICACIÓN CLAVE ---

    if evaluation_item and not replace:
        return evaluation_item[0]
    elif evaluation_item and replace:
        existing_evaluation_list = [item for item in existing_evaluation_list if item not in evaluation_item]

    pipeline_start_time = time.time()
    output = RAGChain.get_rag_chain().invoke(test_case['input'])
    pipeline_end_time = time.time()
    total_pipeline_latency = pipeline_end_time - pipeline_start_time

    llm_performance_metrics = {}
    if hasattr(RAGChain.llm_instance, 'last_call_metrics') and RAGChain.llm_instance.last_call_metrics:
        metrics = RAGChain.llm_instance.last_call_metrics
        llm_performance_metrics = {
            "llm_latency_seconds": metrics.get("latency_seconds"),
            "energy_consumed_wh": metrics.get("energy_consumed_wh"),
            "input_tokens": metrics.get("input_tokens"),
            "output_tokens": metrics.get("output_tokens"),
            "total_tokens": metrics.get("total_tokens"),
            "tokens_per_second": metrics.get("tokens_per_second")
        }

    llm_test_case = LLMTestCase(
        input=test_case['input'],
        actual_output=output['answer'],
        expected_output=test_case['expected_output'],
        retrieval_context=[doc.page_content for doc in output['retrievals']]
    )

    metric_AnswerRelevancyMetric = AnswerRelevancyMetric(threshold=0.7, model=evaluation_model, include_reason=False)
    metric_FaithfulnessMetric = FaithfulnessMetric(threshold=0.7, model=evaluation_model, include_reason=False)
    metric_ContextualRecallMetric = ContextualRecallMetric(threshold=0.7, model=evaluation_model, include_reason=False)
    metric_ContextualPrecisionMetric = ContextualPrecisionMetric(threshold=0.7, model=evaluation_model, include_reason=False)

    evaluation_results = evaluate(
        [llm_test_case],
        [metric_AnswerRelevancyMetric, metric_FaithfulnessMetric, metric_ContextualRecallMetric, metric_ContextualPrecisionMetric]
    )

    evaluation_scores = [{"metric":metric.name, "score":metric.score} for metric in evaluation_results.test_results[0].metrics_data]
    average_score = sum(item['score'] for item in evaluation_scores) / len(evaluation_scores) if evaluation_scores else 0

    evaluation_item = {
        "input": test_case['input'],
        "actual_output": output['answer'],
        "expected_output": test_case['expected_output'],
        "retrieval_context": [doc.page_content for doc in output['retrievals']],
        "context": test_case.get('context'),
        "source": RAGChain.source,
        "pdf_loader": RAGChain.pdf_loader,
        "embedding_model": RAGChain.embedding_model,
        "db_provider": RAGChain.db_provider,
        "llm_model": RAGChain.llm_model,
        "retrieval_strategy": RAGChain.retrieval_strategy,
        "retriever_llm_model": RAGChain.retriever_llm_model,
        "k": RAGChain.k,
        "total_pipeline_latency_seconds": round(total_pipeline_latency, 4),
        "Contextual Precision": round(next((item['score'] for item in evaluation_scores if item['metric'] == 'Contextual Precision'), 0), 3),
        "Contextual Recall": round(next((item['score'] for item in evaluation_scores if item['metric'] == 'Contextual Recall'), 0), 3),
        "Answer Relevancy": round(next((item['score'] for item in evaluation_scores if item['metric'] == 'Answer Relevancy'), 0), 3),
        "Faithfulness": round(next((item['score'] for item in evaluation_scores if item['metric'] == 'Faithfulness'), 0), 3),
        "RAGAs Average": round(average_score, 3)
    }

    evaluation_item.update(llm_performance_metrics)

    existing_evaluation_list.append(evaluation_item)

    with open(evaluation_json, 'w', encoding='utf-8') as f:
        json.dump(existing_evaluation_list, f, indent=4)

    return evaluation_item

In [ ]:
## ----------------------------------------------------------------------------
## FLUJO DE EVALUACIÓN CON DEEPEVAL Y RAGAS:
# Interfaz de evaluación para Gemini Flash 1.5 (VERSIÓN DEFINITIVA)
## ----------------------------------------------------------------------------

import os
import google.generativeai as genai
from deepeval.models import DeepEvalBaseLLM

# Variable global para asegurar que la configuración de la API se haga una sola vez
_gemini_is_configured = False

class CustomGeminiFlash(DeepEvalBaseLLM):
    def __init__(self):
        global _gemini_is_configured

        if not _gemini_is_configured:
            try:
                print("✨ Configurando la API de Google Gemini por primera vez...")
                genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
                _gemini_is_configured = True
            except Exception as e:
                raise ValueError(f"Error CRÍTICO al configurar la API de Gemini. Verifica tu GOOGLE_API_KEY. Error original: {e}")

        self.model = genai.GenerativeModel(model_name="models/gemini-1.5-flash")

    def load_model(self):
        return self.model

    # Versión síncrona (funciona bien)
    def generate(self, prompt: str) -> str:
        response = self.model.generate_content(prompt)
        return response.text

    # --- INICIO DE LA CORRECCIÓN CLAVE ---
    # La versión asíncrona ahora simplemente llama a la versión síncrona.
    # Esto evita el conflicto de bucles asíncronos entre DeepEval y la librería de Google.
    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)
    # --- FIN DE LA CORRECCIÓN CLAVE ---

    def get_model_name(self):
        return "Gemini 1.5 Flash"

In [ ]:
## ----------------------------------------------------------------------------
## SCRIPT DE PRUEBA DEL FLUJO DE EVALUACIÓN - VERSIÓN DEFINITIVA
## MODIFICADO PARA SELECCIONAR EL LLM DEL RETRIEVER
## ----------------------------------------------------------------------------

import os
import json
import time

print("🚀 INICIANDO PRUEBA DEL FLUJO DE EVALUACIÓN (VERSIÓN DEFINITIVA)...")

# 1. Definición de Rutas Absolutas (sin cambios)
BASE_PATH = "/content/drive/MyDrive/Tesis"
PDF_FOLDER_PATH = os.path.join(BASE_PATH, "documents")
DATASET_FOLDER_PATH = os.path.join(BASE_PATH, "datasetEval")
RESULTS_FOLDER_PATH = os.path.join(BASE_PATH, "evaluationResults")

print(f"Buscando PDFs en: {PDF_FOLDER_PATH}")
print(f"Buscando Datasets en: {DATASET_FOLDER_PATH}")
print(f"Guardando resultados en: {RESULTS_FOLDER_PATH}")

# 2. Creamos el modelo de evaluación UNA SOLA VEZ (sin cambios)
print("\n✨ Creando instancia única del modelo de evaluación (CustomGeminiFlash)...")
try:
    evaluation_model = CustomGeminiFlash()
except Exception as e:
    print(f"❌ Error CRÍTICO al crear el modelo de evaluación: {e}")
    raise

# 3. Define las combinaciones que quieres probar.
pdf_filenames_list = ["Creditos Inteligentes.pdf", "Crédito Tradicional.pdf", "Manual de Politicas de Crédito.pdf", "SOP INSTRUCTIVO PARA LA VERIFICACION DE CRÉDITOS.pdf"]
pdf_loader_list = ["PyMuPDF" ,"markerprocessor" , "geminiprocessor", "agenticgeminiprocessor"]
embedding_models_list = ["gemini-embedding-001" , "qwen3-embedding" , "nomic-embed-text"]
db_providers_list = ["mongodb" ,"chromadb"]
llm_model_list = ["gemini-2.5-flash", "qwen3-8b"] # LLMs para la respuesta final

# --- INICIO DE LA MODIFICACIÓN 1: Añadir listas para el retriever ---
# Añade aquí las estrategias que usan un LLM, como 'multiquery' o 'hyde'
retrieval_strategies_list = ["similarity", "multiquery" ,"hyde"]

# Define los LLMs que quieres probar específicamente para el retriever
retriever_llm_model_list = ["gemini-2.5-flash", "qwen3-8b"]

# Lista de control para saber cuándo activar el bucle del retriever LLM
strategies_that_use_llm = ["multiquery", "hyde"]
# --- FIN DE LA MODIFICACIÓN 1 ---

os.makedirs(RESULTS_FOLDER_PATH, exist_ok=True)

# 4. Bucle principal de evaluación
for pdf_filename in pdf_filenames_list:
    path = os.path.join(PDF_FOLDER_PATH, pdf_filename)
    source = os.path.basename(path)
    file_name = os.path.splitext(source)[0]

    dataset_json = os.path.join(DATASET_FOLDER_PATH, f"{file_name}.json")
    evaluation_json = os.path.join(RESULTS_FOLDER_PATH, f"{file_name} - Evaluation.json")

    if not os.path.exists(dataset_json):
        print(f"❌ ADVERTENCIA: Dataset no encontrado en '{dataset_json}'. Saltando este PDF.")
        continue

    if not os.path.exists(evaluation_json):
        with open(evaluation_json, 'w') as f:
            json.dump([], f)

    for pdf_loader in pdf_loader_list:
        for embedding_model in embedding_models_list:
            for db_provider in db_providers_list:
                for llm_model in llm_model_list:
                    for retrieval_strategy in retrieval_strategies_list:

                        # --- INICIO DE LA MODIFICACIÓN 2: Lógica condicional y nuevo bucle ---
                        # Decide sobre qué lista de LLMs de retriever iterar
                        if retrieval_strategy in strategies_that_use_llm:
                            llms_for_retriever_loop = retriever_llm_model_list
                        else:
                            # Si la estrategia no usa LLM (ej. similarity), iteramos sobre un placeholder
                            llms_for_retriever_loop = ["N/A"]

                        # ¡NUEVO BUCLE para el LLM del retriever!
                        for retriever_llm in llms_for_retriever_loop:
                            print("\n" + "=" * 70)
                            print(f"⚙️  Evaluando la combinación:")
                            # El print ahora incluye el LLM del retriever para mayor claridad
                            print(f"    - PDF: {source} | Loader: {pdf_loader} | LLM Principal: {llm_model}")
                            print(f"    - Retriever: {retrieval_strategy} | Retriever LLM: {retriever_llm}")
                            print("=" * 70)

                            # Creamos la instancia del LLM principal
                            llm_instance = MyLLM(model=llm_model, temperature=0.5).get_llm()

                            RAGChain = MyRAGChain(
                                source=source,
                                pdf_loader=pdf_loader,
                                embedding_model=embedding_model,
                                db_provider=db_provider,
                                llm_model=llm_model,
                                k=5,
                                temperature=0.5,
                                retrieval_strategy=retrieval_strategy,
                                retriever_llm_model=retriever_llm, # <-- Pasamos el LLM del retriever
                                llm_instance=llm_instance
                            )

                            with open(dataset_json, 'r', encoding='utf-8') as f:
                                dataset_list = json.load(f)

                            for ii, test_case in enumerate(dataset_list):
                                print(f"   -> Procesando pregunta {ii + 1}/{len(dataset_list)}...")
                                try:
                                    gemini_evaluate_test_case(
                                        RAGChain=RAGChain,
                                        test_case=test_case,
                                        evaluation_model=evaluation_model,
                                        evaluation_json=evaluation_json,
                                        replace=True
                                    )
                                    print(f"   ✅ Pregunta {ii + 1} evaluada con éxito.")
                                except Exception as e:
                                    print(f"   ❌ Error evaluando el caso {ii + 1}: {e}")
                                    continue
                        # --- FIN DE LA MODIFICACIÓN 2 ---

print("\n🎉🎉🎉 ¡FLUJO DE EVALUACIÓN COMPLETADO CON ÉXITO! 🎉🎉🎉")

# EVALUADOR FINAL

In [ ]:
## ----------------------------------------------------------------------------
## SCRIPT DE PRUEBA DEL FLUJO DE EVALUACIÓN - CON CONTADOR DE PROGRESO
## EJECUTA TODAS LAS COMBINACIONES SOBRE 1 DOCUMENTO CON 15 PREGUNTAS
## ----------------------------------------------------------------------------

import os
import json
import time

print("🚀 INICIANDO PRUEBA DEL FLUJO DE EVALUACIÓN (1 Documento x 15 Preguntas)...")

# 1. Definición de Rutas Absolutas (sin cambios)
BASE_PATH = "/content/drive/MyDrive/Tesis"
PDF_FOLDER_PATH = os.path.join(BASE_PATH, "documents")
DATASET_FOLDER_PATH = os.path.join(BASE_PATH, "datasetEval")
RESULTS_FOLDER_PATH = os.path.join(BASE_PATH, "evaluationResults")

print(f"Buscando PDFs en: {PDF_FOLDER_PATH}")
print(f"Buscando Datasets en: {DATASET_FOLDER_PATH}")
print(f"Guardando resultados en: {RESULTS_FOLDER_PATH}")

# 2. Creamos el modelo de evaluación UNA SOLA VEZ (sin cambios)
print("\n✨ Creando instancia única del modelo de evaluación (CustomGeminiFlash)...")
try:
    evaluation_model = CustomGeminiFlash()
except Exception as e:
    print(f"❌ Error CRÍTICO al crear el modelo de evaluación: {e}")
    raise

# 3. Define las combinaciones que quieres probar (sin cambios)

# --- Listas de Componentes LOCALES ---
local_pdf_loader_list = ["PyMuPDF"]
local_embedding_models_list = ["qwen3-embedding", "nomic-embed-text"]
local_db_providers_list = ["chromadb"]
local_llm_model_list = ["qwen3-8b"]
local_retriever_llm_model_list = ["qwen3-8b"]

# --- Listas de Componentes en la NUBE ---
cloud_pdf_loader_list = ["markerprocessor", "geminiprocessor", "agenticgeminiprocessor"]
cloud_embedding_models_list = ["gemini-embedding-001"]
cloud_db_providers_list = ["mongodb"]
cloud_llm_model_list = ["gemini-2.5-flash"]
cloud_retriever_llm_model_list = ["gemini-2.5-flash"]

# --- Configuración General de Estrategias ---
retrieval_strategies_list = ["similarity", "multiquery", "hyde"]
strategies_that_use_llm = ["multiquery", "hyde"]

# --- MODIFICACIÓN CLAVE: SELECCIONA UN ÚNICO DOCUMENTO PARA LA PRUEBA ---
documentos_a_probar = ["SOP INSTRUCTIVO PARA LA VERIFICACION DE CRÉDITOS.pdf"]
print(f"\n🎯 MODO DE PRUEBA ÓPTIMO: Se evaluará únicamente el documento: {documentos_a_probar[0]}")
# --------------------------------------------------------------------------

os.makedirs(RESULTS_FOLDER_PATH, exist_ok=True)

# ----------------------------------------------------------------------------------
# 4. BUCLE 1: EVALUACIÓN DE COMBINACIONES LOCALES
# ----------------------------------------------------------------------------------
print("\n" + "#" * 80)
print(f"### INICIANDO EVALUACIÓN LOCAL para: {documentos_a_probar[0]} ###")
print("#" * 80)

# --- NUEVO: CÁLCULO DEL TOTAL DE COMBINACIONES LOCALES ---
local_retriever_iterations = sum(
    len(local_retriever_llm_model_list) if s in strategies_that_use_llm else 1
    for s in retrieval_strategies_list
)
total_local_combinations = (
    len(local_pdf_loader_list) *
    len(local_embedding_models_list) *
    len(local_db_providers_list) *
    len(local_llm_model_list) *
    local_retriever_iterations
)
local_combination_counter = 0
# --- FIN NUEVO ---

for pdf_filename in documentos_a_probar:
    path = os.path.join(PDF_FOLDER_PATH, pdf_filename)
    source = os.path.basename(path)
    file_name = os.path.splitext(source)[0]
    dataset_json = os.path.join(DATASET_FOLDER_PATH, f"{file_name}.json")
    evaluation_json = os.path.join(RESULTS_FOLDER_PATH, f"{file_name} - Evaluation.json")

    if not os.path.exists(dataset_json): continue
    if not os.path.exists(evaluation_json):
        with open(evaluation_json, 'w') as f: json.dump([], f)

    for pdf_loader in local_pdf_loader_list:
        for embedding_model in local_embedding_models_list:
            for db_provider in local_db_providers_list:
                for llm_model in local_llm_model_list:
                    for retrieval_strategy in retrieval_strategies_list:
                        if retrieval_strategy in strategies_that_use_llm:
                            llms_for_retriever_loop = local_retriever_llm_model_list
                        else:
                            llms_for_retriever_loop = ["N/A"]
                        for retriever_llm in llms_for_retriever_loop:
                            # --- NUEVO: INCREMENTAR Y MOSTRAR CONTADOR ---
                            local_combination_counter += 1
                            print("\n" + "=" * 70)
                            print(f"--- Procesando Combinación LOCAL {local_combination_counter} de {total_local_combinations} ---")
                            # --- FIN NUEVO ---

                            print(f"⚙️  [LOCAL] Evaluando la combinación:")
                            print(f"    - PDF: {source} | Loader: {pdf_loader} | LLM Principal: {llm_model}")
                            print(f"    - Embedding: {embedding_model} | DB: {db_provider}")
                            print(f"    - Retriever: {retrieval_strategy} | Retriever LLM: {retriever_llm}")
                            print("=" * 70)

                            llm_instance = MyLLM(model=llm_model, temperature=0.5).get_llm()
                            # ... (resto del código sin cambios)
                            RAGChain = MyRAGChain(source=source, pdf_loader=pdf_loader, embedding_model=embedding_model, db_provider=db_provider, llm_model=llm_model, k=5, temperature=0.5, retrieval_strategy=retrieval_strategy, retriever_llm_model=retriever_llm, llm_instance=llm_instance)
                            with open(dataset_json, 'r', encoding='utf-8') as f: dataset_list = json.load(f)
                            for ii, test_case in enumerate(dataset_list):
                                print(f"   -> Procesando pregunta {ii + 1}/{len(dataset_list)}...")
                                try:
                                    gemini_evaluate_test_case(RAGChain=RAGChain, test_case=test_case, evaluation_model=evaluation_model, evaluation_json=evaluation_json, replace=True)
                                    print(f"   ✅ Pregunta {ii + 1} evaluada con éxito.")
                                except Exception as e:
                                    print(f"   ❌ Error evaluando el caso {ii + 1}: {e}")
                                    continue

# ----------------------------------------------------------------------------------
# 5. BUCLE 2: EVALUACIÓN DE COMBINACIONES EN LA NUBE
# ----------------------------------------------------------------------------------
print("\n" + "#" * 80)
print(f"### INICIANDO EVALUACIÓN EN LA NUBE para: {documentos_a_probar[0]} ###")
print("#" * 80)

# --- NUEVO: CÁLCULO DEL TOTAL DE COMBINACIONES EN LA NUBE ---
cloud_retriever_iterations = sum(
    len(cloud_retriever_llm_model_list) if s in strategies_that_use_llm else 1
    for s in retrieval_strategies_list
)
total_cloud_combinations = (
    len(cloud_pdf_loader_list) *
    len(cloud_embedding_models_list) *
    len(cloud_db_providers_list) *
    len(cloud_llm_model_list) *
    cloud_retriever_iterations
)
cloud_combination_counter = 0
# --- FIN NUEVO ---

for pdf_filename in documentos_a_probar:
    path = os.path.join(PDF_FOLDER_PATH, pdf_filename)
    source = os.path.basename(path)
    file_name = os.path.splitext(source)[0]
    dataset_json = os.path.join(DATASET_FOLDER_PATH, f"{file_name}.json")
    evaluation_json = os.path.join(RESULTS_FOLDER_PATH, f"{file_name} - Evaluation.json")
    if not os.path.exists(dataset_json): continue

    for pdf_loader in cloud_pdf_loader_list:
        for embedding_model in cloud_embedding_models_list:
            for db_provider in cloud_db_providers_list:
                for llm_model in cloud_llm_model_list:
                    for retrieval_strategy in retrieval_strategies_list:
                        if retrieval_strategy in strategies_that_use_llm:
                            llms_for_retriever_loop = cloud_retriever_llm_model_list
                        else:
                            llms_for_retriever_loop = ["N/A"]
                        for retriever_llm in llms_for_retriever_loop:
                            # --- NUEVO: INCREMENTAR Y MOSTRAR CONTADOR ---
                            cloud_combination_counter += 1
                            print("\n" + "=" * 70)
                            print(f"--- Procesando Combinación NUBE {cloud_combination_counter} de {total_cloud_combinations} ---")
                            # --- FIN NUEVO ---

                            print(f"⚙️  [NUBE] Evaluando la combinación:")
                            print(f"    - PDF: {source} | Loader: {pdf_loader} | LLM Principal: {llm_model}")
                            print(f"    - Embedding: {embedding_model} | DB: {db_provider}")
                            print(f"    - Retriever: {retrieval_strategy} | Retriever LLM: {retriever_llm}")
                            print("=" * 70)

                            llm_instance = MyLLM(model=llm_model, temperature=0.5).get_llm()
                            # ... (resto del código sin cambios)
                            RAGChain = MyRAGChain(source=source, pdf_loader=pdf_loader, embedding_model=embedding_model, db_provider=db_provider, llm_model=llm_model, k=5, temperature=0.5, retrieval_strategy=retrieval_strategy, retriever_llm_model=retriever_llm, llm_instance=llm_instance)
                            with open(dataset_json, 'r', encoding='utf-8') as f: dataset_list = json.load(f)
                            for ii, test_case in enumerate(dataset_list):
                                print(f"   -> Procesando pregunta {ii + 1}/{len(dataset_list)}...")
                                try:
                                    gemini_evaluate_test_case(RAGChain=RAGChain, test_case=test_case, evaluation_model=evaluation_model, evaluation_json=evaluation_json, replace=True)
                                    print(f"   ✅ Pregunta {ii + 1} evaluada con éxito.")
                                except Exception as e:
                                    print(f"   ❌ Error evaluando el caso {ii + 1}: {e}")
                                    continue

print("\n🎉🎉🎉 ¡FLUJO DE EVALUACIÓN ÓPTIMO COMPLETADO CON ÉXITO! 🎉🎉🎉")

Hyde corregido


In [ ]:
from langchain_core.runnables import RunnablePassthrough, RunnableMap
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate


class MyRAGChain:

    def __init__(self, source: str, pdf_loader: str, embedding_model: str, db_provider: str, llm_model: str, k: int = 5, temperature: float = 0.5,
                 retrieval_strategy: str = "similarity",
                 retriever_llm_model: str = "gemini-2.5-flash",
                 retriever_llm_temperature: float = 0.0,
                 llm_instance = None): # <-- El parámetro para la instancia del LLM
        self.source = source
        self.pdf_loader = pdf_loader
        self.embedding_model = embedding_model
        self.db_provider = db_provider
        self.llm_model = llm_model
        self.k = k
        self.temperature = temperature
        self.retrieval_strategy = retrieval_strategy
        self.retriever_llm_model = retriever_llm_model
        self.retriever_llm_temperature = retriever_llm_temperature
        self.llm_instance = llm_instance

    def get_rag_chain(self):

        def format_docs(docs):
            return "\n\n".join(doc.page_content for doc in docs)

        # --- Plantilla para la respuesta final (común a todas las estrategias) ---
        RAG_TEMPLATE = """
        You are an assistant for question-answering tasks.
        Use only the following pieces of retrieved context to answer the question.
        If you don't know the answer, just say that you don't know.

        <context>
        {context}
        </context>

        Answer the following question:

        {question}"""
        rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

        # --- Configuración de Embedder, VectorStore y Retriever ---
        Embedder = MyEmbedder(model=self.embedding_model)
        VectorStore = MyVectorStore(provider=self.db_provider, Embedder=Embedder)

        # Si la estrategia es 'hyde', el retriever base sigue siendo de similitud,
        # ya que HyDE actúa *antes* del retriever.
        base_retrieval_strategy_for_hyde = "similarity" if self.retrieval_strategy == "hyde" else self.retrieval_strategy

        retriever = MyRetriever(
            k=self.k,
            source=self.source,
            parser=self.pdf_loader,
            VectorStore=VectorStore,
            retrieval_strategy=base_retrieval_strategy_for_hyde,
            llm_model=self.retriever_llm_model,
            temperature=self.retriever_llm_temperature
        ).get_retriever()

        # --- Configuración del LLM ---
        # Usa la instancia pre-creada si se proporciona, si no, crea una nueva.
        if self.llm_instance:
            llm = self.llm_instance
        else:
            llm = MyLLM(model=self.llm_model, temperature=self.temperature).get_llm()

        # --- LÓGICA CONDICIONAL PARA CADA ESTRATEGIA DE RETRIEVAL ---

        if self.retrieval_strategy == "hyde":
            # --------------------------------------------------------------------
            # INICIO DE LA LÓGICA CORREGIDA PARA 'hyde'
            # --------------------------------------------------------------------
            print("Estrategia de Retrieval: HyDE (Hypothetical Document Embeddings)")

            # Plantilla para generar la respuesta hipotética
            HYDE_TEMPLATE = """
            Escribe un pasaje conciso que responda a la siguiente pregunta.
            Describe la respuesta como si proviniera de una fuente de información fiable.
            No digas "según la fuente" o frases similares.

            Pregunta: {question}
            Pasaje:
            """
            hyde_prompt = ChatPromptTemplate.from_template(HYDE_TEMPLATE)

            # LLM para el retriever (usamos los parámetros específicos para ello)
            retriever_llm = MyLLM(model=self.retriever_llm_model, temperature=self.retriever_llm_temperature).get_llm()

            # Cadena para generar la respuesta hipotética
            hyde_chain = hyde_prompt | retriever_llm | StrOutputParser()

            # Cadena RAG completa para HyDE
            full_chain = (
                {
                    # La clave 'retrievals' ahora depende de un paso intermedio.
                    "retrievals": RunnablePassthrough.assign(
                        # 1. Genera la respuesta hipotética a partir de la pregunta original.
                        hypothetical_answer=hyde_chain
                    # 2. Usa esa respuesta hipotética para invocar al retriever.
                    ) | (lambda x: retriever.invoke(x["hypothetical_answer"])),
                    # Pasamos la pregunta original sin cambios.
                    "question": lambda x: x["question"],
                }
                | RunnableMap(
                    {
                        "retrievals": lambda inputs: inputs["retrievals"],
                        "answer": (
                            {
                                # 3. Usa los documentos recuperados y la pregunta ORIGINAL.
                                "context": lambda inputs: format_docs(inputs["retrievals"]),
                                "question": lambda inputs: inputs["question"],
                            }
                            # 4. Genera la respuesta final con el LLM principal.
                            | rag_prompt
                            | llm
                            | StrOutputParser()
                        ),
                    }
                )
            )
            # --------------------------------------------------------------------
            # FIN DE LA LÓGICA CORREGIDA
            # --------------------------------------------------------------------
        else:
            # Esta es la lógica original para 'similarity' y 'multiquery', que no cambia.
            full_chain = (
                {
                    "retrievals": retriever,
                    "question": RunnablePassthrough(),
                }
                | RunnableMap(
                    {
                        "retrievals": lambda inputs: inputs["retrievals"],
                        "answer": (
                            {
                                "context": lambda inputs: format_docs(inputs["retrievals"]),
                                "question": lambda inputs: inputs["question"],
                            }
                            | rag_prompt
                            | llm
                            | StrOutputParser()
                        ),
                    }
                )
            )

        # La salida es la misma para todas las cadenas, garantizando compatibilidad.
        return full_chain | (lambda outputs: {"retrievals": outputs["retrievals"], "answer": outputs["answer"]})

In [ ]:
## ----------------------------------------------------------------------------
## IMPORTACIÓN DE LAS LIBRERÍAS NECESARIAS PARA EL ANÁLISIS DE RESULTADOS
## ----------------------------------------------------------------------------
import os
import json
import pandas as pd
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

In [ ]:
## ----------------------------------------------------------------------------
## DATAFRAME DE RESULTADOS:
# Construcción de un DataFrame con los resultados de la evaluación de los
# pipelines de RAG para facilitar su análisis.
## ----------------------------------------------------------------------------

results_df_list = []

# For all the pdfs
# Available PDFs: 'Attention Is All You Need.pdf', 'CODIMUR 50.pdf', 'Orden Ayudas Publicas.pdf', 'PPT Sostenibilidad.pdf', 'UOC Ciencia de Datos.pdf'
pdfs_list = [r"../PDFs/Attention Is All You Need.pdf", r"../PDFs/CODIMUR 50.pdf", r"../PDFs/Orden Ayudas Publicas.pdf", r"../PDFs/PPT Sostenibilidad.pdf", r"../PDFs/UOC Ciencia de Datos.pdf"]
for ii, pdf in enumerate(pdfs_list):
    file_name = os.path.basename(pdf).split(".")[0]

    # Load the evaluation test cases
    dataset_json = f"../DeepEval/{file_name} - Dataset.json"
    with open(dataset_json, 'r') as f:
        dataset_list = json.load(f)

    # Load the evaluation results
    reults_json = f"../DeepEval/{file_name} - Evaluation.json"
    with open(reults_json, 'r') as f:
        results_list = json.load(f)

    # Build the DataFrame
    df = pd.DataFrame(results_list)

    # Add the question_id
    for jj, question in enumerate(dataset_list):
        df.loc[df['input'] == question['input'], 'question_id'] = f'{ii+1}.{jj+1}'

    # Scale to Percentage
    df['Contextual Precision'] = df['Contextual Precision'] * 100
    df['Contextual Recall'] = df['Contextual Recall'] * 100
    df['Answer Relevancy'] = df['Answer Relevancy'] * 100
    df['Faithfulness'] = df['Faithfulness'] * 100
    df['RAGAs Average'] = df['RAGAs Average'] * 100

    # Tag the environment: [Local, Cloud, Mixed]
    df['Environment'] = df[['pdf_loader', 'embedding_model', 'db_provider', 'llm_model']].apply(
        lambda x: 'Local' if (x.iloc[0] in ['PyMuPDF', 'AzureDocIntLocal'] and x.iloc[1] in ['nomic-embed-text', 'snowflake-arctic-embed-335m'] and x.iloc[2] in ['chromadb'] and x.iloc[3] in ['gemma2-9b', 'llama3-1-8b', 'phi3-5'])
        else 'Cloud' if (x.iloc[0] in ['UnstructuredCloud', 'LlamaParse'] and x.iloc[1] in ['text-embedding-3-large', 'voyage-3'] and x.iloc[2] in ['mongodb'] and x.iloc[3] in ['gpt-4o', 'claude-3-5-sonnet', 'gemini-1.5-pro'])
        else 'Mixed', axis=1)

    # Append the DataFrame to the list
    df.drop(columns=['context'], inplace=True)
    results_df_list.append(df)

# Save the DataFrame
results_df = pd.concat(results_df_list)
results_df.to_csv("../Results/Results.csv", index=False)

In [ ]:
## ----------------------------------------------------------------------------
## DATAFRAME DE RESULTADOS:
# Muestra del DataFrame con los resultados de la evaluación de los pipelines de RAG.
## ----------------------------------------------------------------------------
df = pd.read_csv("../Results/Results.csv", dtype={'question_id': str})
df.sample(5)

In [ ]:
## ----------------------------------------------------------------------------
## DATAFRAME DE RESULTADOS:
# Validamos que todos los tests se hayan ejecutado correctamente (11520).
## ----------------------------------------------------------------------------
df = pd.read_csv("../Results/Results.csv", dtype={'question_id': str})
df.info()

In [ ]:
## ----------------------------------------------------------------------------
## COMPARACIÓN DE RENDIMIENTO GLOBAL POR ENTORNO - POR TEST
## ----------------------------------------------------------------------------
# Cargar el DataFrame de resultados
df = pd.read_csv("../Results/Results.csv", dtype={'question_id': str})

# Filtrar los datos para incluir solo los entornos Local y Cloud
filtered_df = df[df['Environment'].isin(['Local', 'Cloud'])]

# Crear un Boxplot del promedio separado por entorno
plt.figure(figsize=(12, 10))
sns.boxplot(data=filtered_df, x='Environment', y='RAGAs Average', hue='Environment', palette="Set2", dodge=False, linewidth=2.5)
plt.title('Comparación del Rendimiento General\nentre Entornos Local y Cloud', fontsize=30, pad=20)
plt.ylabel('RAGAs Average (%)', fontsize=30)
plt.xticks(fontsize=24)
plt.yticks(fontsize=24)
plt.xlabel('')
plt.legend([],[], frameon=False)

# Ajustes generales de diseño
plt.tight_layout()

# Guardar la figura
plt.savefig("../Results/comparacion_rendimiento_global_entornos_por_test.png")

# Mostrar la figura
plt.show()

In [ ]:
## ----------------------------------------------------------------------------
## COMPARACIÓN DE RENDIMIENTO GLOBAL POR ENTORNO - POR PIPELINE
## ----------------------------------------------------------------------------
# Cargar el DataFrame de resultados
df = pd.read_csv("../Results/Results.csv", dtype={'question_id': str})

# Filtrar los datos para incluir solo los entornos Local y Cloud
filtered_df = df[df['Environment'].isin(['Local', 'Cloud'])]

# Seleccionar las métricas relevantes
metrics = ['Contextual Precision', 'Contextual Recall', 'Answer Relevancy', 'Faithfulness', 'RAGAs Average']

# Agrupar por pipeline
grouped_df = filtered_df.groupby(['pdf_loader', 'embedding_model', 'db_provider', 'llm_model', 'Environment'])[metrics].mean().reset_index()

# Crear un Boxplot del promedio separado por entorno
plt.figure(figsize=(12, 10))
sns.boxplot(data=grouped_df, x='Environment', y='RAGAs Average', hue='Environment', palette="Set2", dodge=False, linewidth=2.5)
plt.title('Comparación del Rendimiento General\nentre Entornos Local y Cloud', fontsize=30, pad=20)
plt.ylabel('RAGAs Average (%)', fontsize=30)
plt.xticks(fontsize=24)
plt.yticks(fontsize=24)
plt.xlabel('')
plt.legend([],[], frameon=False)

# Ajustes generales de diseño
plt.tight_layout()

# Guardar la figura
plt.savefig("../Results/comparacion_rendimiento_global_entornos_por_pipeline.png")

# Mostrar la figura
plt.show()

In [ ]:
## ----------------------------------------------------------------------------
## MATRIZ DE CORRELACIÓN DE MÉTRICAS RAGAS
## ----------------------------------------------------------------------------
# Cargar el DataFrame de resultados
df = pd.read_csv("../Results/Results.csv", dtype={'question_id': str})

# Seleccionar las métricas relevantes
metrics_without_average = ['Contextual Precision', 'Contextual Recall', 'Answer Relevancy', 'Faithfulness']

# Calcular la matriz de correlación
correlation_matrix = df[metrics_without_average].corr(method='pearson')

# Crear un heatmap para la matriz de correlación
plt.figure(figsize=(12, 10))
heatmap = sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap=sns.color_palette("ch:s=-.2,r=.6", as_cmap=True),
    cbar=True,
    linewidths=0.5,
    linecolor='white',
    annot_kws={"size": 20},
    alpha=0.8,
    vmin=-1,
    vmax=1
)

# Adjust labels to have new lines instead of spaces
labels = [label.replace(" ", "\n") for label in correlation_matrix.columns]
heatmap.set_xticklabels(labels, fontsize=24, rotation=45)
heatmap.set_yticklabels(labels, fontsize=24, rotation=0)

colorbar = heatmap.collections[0].colorbar
colorbar.ax.tick_params(labelsize=20)
colorbar.set_ticks([-1, -0.5, 0, 0.5, 1])

plt.title('Matriz de Correlación de Métricas', fontsize=30, pad=20)

# Ajustes generales de diseño
plt.tight_layout()

# Guardar la figura
plt.savefig("../Results/matriz_correlacion_metricas.png")

# Mostrar la figura
plt.show()

In [ ]:
## ----------------------------------------------------------------------------
## DENSAIDAD DE MÉTRICAS RAGAS POR ENTORNO - POR TEST
## ----------------------------------------------------------------------------
# Cargar el DataFrame de resultados
df = pd.read_csv("../Results/Results.csv", dtype={'question_id': str})

# Filtrar los datos para incluir solo los entornos Local y Cloud
filtered_df = df[df['Environment'].isin(['Local', 'Cloud'])]

# Seleccionar las métricas relevantes
metrics_without_average = ['Contextual Precision', 'Contextual Recall', 'Answer Relevancy', 'Faithfulness']

# Formatear los datos para el Violin Plot
melted_df = pd.melt(
    filtered_df,
    id_vars=['Environment'],
    value_vars=metrics_without_average,
    var_name='Métrica',
    value_name='Valor'
)

# Crear Violin Plots
plt.figure(figsize=(24, 10))
sns.violinplot(data=melted_df, x='Métrica', y='Valor', hue='Environment', palette='Set2', split=True, inner=None, linewidth=1.2)
plt.title('Distribución de Métricas RAGAs por Entorno', fontsize=30, pad=20)
plt.ylabel('Valor (%)', fontsize=30)
plt.xlabel('')
plt.xticks(fontsize=24)
plt.yticks(fontsize=24)
plt.ylim(0, 100)
plt.grid(True)
plt.legend(fontsize=24, loc='upper left')

# Adjust labels to have new lines instead of spaces
labels = [label.replace(" ", "\n") for label in melted_df['Métrica'].unique()]
plt.gca().set_xticklabels(labels)

# Ajustes generales de diseño
plt.tight_layout()

# Guardar la figura
plt.savefig("../Results/densidad_metricas_ragas_por_entorno_test.png")

# Mostrar la figura
plt.show()

In [ ]:
## ----------------------------------------------------------------------------
## DENSAIDAD DE MÉTRICAS RAGAS POR ENTORNO - POR PIPELINE
## ----------------------------------------------------------------------------
# Cargar el DataFrame de resultados
df = pd.read_csv("../Results/Results.csv", dtype={'question_id': str})

# Filtrar los datos para incluir solo los entornos Local y Cloud
filtered_df = df[df['Environment'].isin(['Local', 'Cloud'])]

# Seleccionar las métricas relevantes
metrics_without_average = ['Contextual Precision', 'Contextual Recall', 'Answer Relevancy', 'Faithfulness']

# Agrupar por pipeline
grouped_df = filtered_df.groupby(['pdf_loader', 'embedding_model', 'db_provider', 'llm_model', 'Environment'])[metrics_without_average].mean().reset_index()

# Formatear los datos para el Violin Plot
melted_df = pd.melt(
    grouped_df,
    id_vars=['Environment'],
    value_vars=metrics_without_average,
    var_name='Métrica',
    value_name='Valor'
)

# Crear Violin Plots
plt.figure(figsize=(24, 10))
sns.violinplot(data=melted_df, x='Métrica', y='Valor', hue='Environment', palette='Set2', split=True, inner=None, linewidth=1.2)
plt.title('Distribución de Métricas RAGAs por Entorno', fontsize=30, pad=20)
plt.ylabel('Valor (%)', fontsize=30)
plt.xlabel('')
plt.xticks(fontsize=24)
plt.yticks(fontsize=24)
plt.ylim(0, 100)
plt.grid(True)
plt.legend(fontsize=24, loc='upper left')

# Adjust labels to have new lines instead of spaces
labels = [label.replace(" ", "\n") for label in melted_df['Métrica'].unique()]
plt.gca().set_xticklabels(labels)

# Ajustes generales de diseño
plt.tight_layout()

# Guardar la figura
plt.savefig("../Results/densidad_metricas_ragas_por_entorno_pipeline.png")

# Mostrar la figura
plt.show()

In [ ]:
## ----------------------------------------------------------------------------
## ANÁLISIS DE COMPONENTES INDIVIDUALES
## ----------------------------------------------------------------------------
# Cargar el DataFrame de resultados
df = pd.read_csv("../Results/Results.csv", dtype={'question_id': str})

# Seleccionar las métricas relevantes
numeric_columns = ['Contextual Precision', 'Contextual Recall', 'Answer Relevancy', 'Faithfulness', 'RAGAs Average']

# Calcular promedios por componente
loader_averages = df.groupby('pdf_loader')[numeric_columns].mean()
embedding_averages = df.groupby('embedding_model')[numeric_columns].mean()
db_averages = df.groupby('db_provider')[numeric_columns].mean()
llm_averages = df.groupby('llm_model')[numeric_columns].mean()

# Dividir componentes por entorno
cloud_loaders = ['UnstructuredCloud', 'LlamaParse']
local_loaders = ['PyMuPDF', 'AzureDocIntLocal']

cloud_embeddings = ['text-embedding-3-large', 'voyage-3']
local_embeddings = ['nomic-embed-text', 'snowflake-arctic-embed-335m']

cloud_dbs = ['mongodb']
local_dbs = ['chromadb']

cloud_llms = ['gpt-4o', 'claude-3-5-sonnet', 'gemini-1.5-pro']
local_llms = ['gemma2-9b', 'llama3-1-8b', 'phi3-5']

# Calcular promedios por entorno para cada tipo de componente
cloud_average_loaders = loader_averages.loc[cloud_loaders].mean()
local_average_loaders = loader_averages.loc[local_loaders].mean()

cloud_avg_embeddings = embedding_averages.loc[cloud_embeddings].mean()
local_avg_embeddings = embedding_averages.loc[local_embeddings].mean()

cloud_avg_dbs = db_averages.loc[cloud_dbs].mean()
local_avg_dbs = db_averages.loc[local_dbs].mean()

cloud_avg_llms = llm_averages.loc[cloud_llms].mean()
local_avg_llms = llm_averages.loc[local_llms].mean()

# Configuración de estilo general
sns.set_theme(style="whitegrid")

# Función auxiliar para cada subplot
def plot_subplot(ax, data, cloud_avg, local_avg, title, bar_width, ylim):
    # Barras para cada componente
    components = data.index
    num_metrics = len(numeric_columns)
    for idx, component in enumerate(components):
        ax.bar([x + idx * bar_width for x in range(num_metrics)], data.loc[component], width=bar_width, label=component, alpha=0.7)

    # Fondo destacado para la métrica de promedio
    ax.axvspan(num_metrics - 1 - bar_width * len(components) / 2, num_metrics - 1 + bar_width * len(components), color='gold', alpha=0.4, zorder=0)

    # Líneas promedio para Cloud y Local
    obscure_colors = ['#696969', '#FF6347']
    ax.plot(range(num_metrics), cloud_avg, linestyle='--', color=obscure_colors[0], linewidth=2.5, label='Cloud Avg')
    ax.plot(range(num_metrics), local_avg, linestyle='--', color=obscure_colors[1], linewidth=2.5, label='Local Avg')

    # Configurar el subplot
    ax.set_title(title, fontsize=40, pad=20)
    ax.set_ylim(*ylim)
    ax.set_xticks([x + (len(components) - 1) * bar_width / 2 for x in range(num_metrics)])
    ax.set_xticklabels([label.replace(" ", "\n") for label in numeric_columns], rotation=45, ha='right', fontsize=24)
    ax.set_ylabel('Valor (%)', fontsize=40)
    ax.yaxis.grid(True)
    ax.xaxis.grid(False)
    ax.legend(loc='upper left' if title != 'Embedding Models' else 'lower right', fontsize=24)
    ax.tick_params(axis='y', labelsize=24)

# Crear un conjunto de gráficos 2x2
fig, axs = plt.subplots(2, 2, figsize=(24, 24))
plot_subplot(axs[0, 0], loader_averages, cloud_average_loaders, local_average_loaders, 'PDF Loaders', bar_width=0.15, ylim=(55, 100))
plot_subplot(axs[0, 1], embedding_averages, cloud_avg_embeddings, local_avg_embeddings, 'Embedding Models', bar_width=0.15, ylim=(10, 100))
plot_subplot(axs[1, 0], db_averages, cloud_avg_dbs, local_avg_dbs, 'Vector DBs', bar_width=0.3, ylim=(55, 100))
plot_subplot(axs[1, 1], llm_averages, cloud_avg_llms, local_avg_llms, 'LLMs', bar_width=0.1, ylim=(55, 100))
fig.suptitle('Análisis de Componentes Individuales', fontsize=40)

# Ajustes generales de diseño
plt.tight_layout(rect=[0, 0, 1, 0.96])

# Guardar la figura
plt.savefig("../Results/analisis_componentes_individuales.png")

# Mostrar la figura
plt.show()

In [ ]:
## ----------------------------------------------------------------------------
## INFLUENCIA DEL TIPO DE PDF Y EL PDF LOADER EN EL RAG
## ----------------------------------------------------------------------------
# Cargar el DataFrame de resultados
df = pd.read_csv("../Results/Results.csv", dtype={'question_id': str})

# Seleccionar las métricas relevantes
numeric_columns = ['Contextual Precision', 'Contextual Recall', 'Answer Relevancy', 'Faithfulness', 'RAGAs Average']

# Agrupar datos para calcular los promedios
env_averges = df.groupby(['source', 'Environment'])[numeric_columns].mean().reset_index()
loader_averages = df.groupby(['source',  'pdf_loader'])[numeric_columns].mean().reset_index()

# Crear una cuadrícula 2x3
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Iterar sobre los PDFs
pdfs = df['source'].unique().tolist()
for idx, pdf in enumerate(pdfs):
    row, col = divmod(idx, 3)
    ax = axes[row, col]

    # Dibujar las líneas horizontales para los entornos
    env_means = env_averges[env_averges['source'] == pdf].groupby('Environment')['RAGAs Average'].mean()
    for y, env in zip([-2, -1, 0], env_means.items()):
        ax.hlines(y=env[1], xmin=-0.1, xmax=0.1, color='black', linewidth=2)
        ax.text(0.15, env[1], env[0], va='center', ha='left', fontsize=12)

    # Dibujar las líneas horizontales para los loaders
    colors = sns.color_palette("Set2", 4)
    loader_means = loader_averages[loader_averages['source'] == pdf].groupby('pdf_loader')['RAGAs Average'].mean()
    for y, (loader, color) in zip([2, 3, 4, 5], zip(loader_means.items(), colors)):
        ax.hlines(y=loader[1], xmin=0.9, xmax=1.1, color=color, linewidth=2)
        # ax.text(1.15, loader[1], loader[0], va='center', ha='left', fontsize=8, color=color)

    # Añadir etiquetas discretas para Environments y Loaders dentro de cada subplot
    ax.text(0, 48, "Environments", va='center', ha='center', fontsize=12, color='gray')
    ax.text(1, 48, "PDF Loaders", va='center', ha='center', fontsize=12, color='gray')

    # Configurar el subplot
    ax.set_xlim(-0.5, 1.5)
    ax.set_ylim(50, 100)
    ax.set_xticks([])
    ax.set_title(f"PDF: {pdf}", fontsize=14)
    ax.axvline(x=0.5, color='black', linestyle='--', linewidth=0.8)

# Añadir leyenda para los loaders en el último subplot vacío
legend_elements = [Line2D([0], [0], color=color, lw=2, label=loader) for loader, color in zip(loader_means.index, colors)]
axes[1, 2].legend(handles=legend_elements, loc='upper left', fontsize=14)
axes[1, 2].axis('off')

# Añadir etiquetas
fig.text(0.04, 0.5, 'RAGAs Average (%)', va='center', rotation='vertical', fontsize=16)
fig.suptitle('Influencia del Tipo de PDF en el RAG', fontsize=20)

# Ajustes generales de diseño
plt.tight_layout(rect=[0.05, 0.1, 1, 0.96])

# Guardar la figura
plt.savefig("../Results/influencia_tipo_pdf_loader.png")

# Mostrar la figura
plt.show()

In [ ]:
## ----------------------------------------------------------------------------
## LOLLIPOP PLOT PARA LOS 5 PEORES RESULTADOS
## ----------------------------------------------------------------------------
# Cargar el DataFrame de resultados
df = pd.read_csv("../Results/Results.csv", dtype={'question_id': str})

# Define las métricas a analizar
metrics = ['Contextual Precision', 'Contextual Recall', 'Answer Relevancy', 'Faithfulness', 'RAGAs Average']

# Crear subplots de 2x3
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, metric in enumerate(metrics):
    # Agrupar por question_id y calcular el promedio de la métrica actual
    avg_metric_per_question = df.groupby('question_id')[metric].mean()

    # Identificar las 5 peores preguntas basadas en el promedio de la métrica
    top_5_worst_questions = avg_metric_per_question.nsmallest(5).index

    # Filtrar el DataFrame para incluir solo las preguntas seleccionadas
    worst_questions_df = df[df['question_id'].isin(top_5_worst_questions)]

    # Calcular el promedio de la métrica para cada entorno
    comparison_df = worst_questions_df.groupby(['question_id', 'Environment'])[metric].mean().unstack()

    # Dibujar el diagrama lollipop para la métrica actual (Cloud vs Local)
    ax = axes[i]
    colors = sns.color_palette("Set2", 5)
    for question_id, color in zip(comparison_df.index, colors):
        cloud_score = comparison_df.loc[question_id, 'Cloud']
        local_score = comparison_df.loc[question_id, 'Local']
        ax.plot([1, 2], [local_score, cloud_score], marker='o', markersize=10, linewidth=2.5, label=f"Q{question_id}", color=color)

    # Ajustes estéticos
    ax.set_xticks([1, 2])
    ax.set_xticklabels(['Local', 'Cloud'], fontsize=18)
    ax.set_ylabel(f"(%)", fontsize=18)
    ax.set_title(f"{metric}", fontsize=20)
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    ax.legend(title="Questions", fontsize=16, title_fontsize=16, loc='upper left' if metric != 'Faithfulness' else 'lower left')
    ax.set_ylim(0, 100)

# Eliminar subplots no utilizados si hay menos métricas
if len(metrics) < len(axes):
    for j in range(len(metrics), len(axes)):
        fig.delaxes(axes[j])

# Añadir título general
fig.suptitle('Comparación de Métricas para las 5 Consultas con Menor Rendimiento', fontsize=26)

# Ajustes generales de diseño
plt.tight_layout()

# Guardar la figura
plt.savefig("../Results/comparacion_metricas_peores_resultados.png")

# Mostrar la figura
plt.show()

In [ ]:
## ----------------------------------------------------------------------------
## SUNBURST CHART DE RAGAS AVERAGE POR COMPONENTES
## ----------------------------------------------------------------------------
import plotly.io as pio
pio.renderers.default = 'notebook'

# Cargar el DataFrame de resultados
df = pd.read_csv("../Results/Results.csv", dtype={'question_id': str})

# Filtrar los datos para incluir solo los entornos Local y Cloud
filtered_df = df[df['Environment'].isin(['Local', 'Cloud'])]

# Preparar datos para Sunburst
sunburst_data = filtered_df.groupby(['Environment', 'pdf_loader', 'db_provider', 'embedding_model', 'llm_model'])['RAGAs Average'].mean().reset_index()

# Crear el gráfico Sunburst
fig = px.sunburst(
    sunburst_data,
    path=['Environment', 'pdf_loader', 'db_provider', 'embedding_model', 'llm_model'],
    values='RAGAs Average',
    color='RAGAs Average',
    color_continuous_scale='Blues',
    title='Sunburst Chart of RAGAs Average by Environment and Components'
)

# Personalizar la fuente y el diseño
fig.update_layout(
    font=dict(
        family="Arial, sans-serif",
        size=20,
        color="black"
    ),
    title_font=dict(
        family="Arial, sans-serif",
        size=30,
        color="black"
    ),
    title_x=0.5,
    coloraxis_colorbar=dict(
        title=""
    ),
    width=1000,
    height=800
)

# Mostrar el gráfico
fig.show()